In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Dados de exemplo: vendas mensais 
data = [
    ("2024-01", 100000),
    ("2024-02", 105000),
    ("2024-03", 315000),  # simulação de aumento suspeito
    ("2024-04", 110000),
]

df = spark.createDataFrame(data, ["mes", "vendas"])

# Calcular variação percentual mês a mês
window = Window.orderBy("mes")
df_validado = df.withColumn(
    "vendas_anterior", F.lag("vendas").over(window)
).withColumn(
    "variacao_pct", 
    ((F.col("vendas") - F.col("vendas_anterior")) / F.col("vendas_anterior") * 100)
)

# Definir limite de alerta
THRESHOLD = 150
anomalias = df_validado.filter(F.abs(F.col("variacao_pct")) > THRESHOLD)

# Validação
if anomalias.count() > 0:
    print("🚨 ALERTA: VARIAÇÕES ANORMAIS DETECTADAS!")
    anomalias.select("mes", "vendas", "vendas_anterior", "variacao_pct").show()
    raise Exception("Pipeline bloqueada - revisar dados antes de prosseguir")
else:
    print("✅ VALIDAÇÃO OK: DADOS DENTRO DO ESPERADO")
    # Continua o pipeline normalmente
    df.write.mode("overwrite").saveAsTable("gold.vendas_mensais")